# Context Engineering - Experience it Live

## Context Rot: Why More Context Can Make an Agent Worse

Imagine you have a banking system running in production.

Your payment service suddenly starts throwing intermittent `5xx` errors and latency is increasing. You need to quickly understand what is going wrong and identify the root cause.

You already have an AI agent that can help with the investigation.

The agent has access to multiple tools to analyse things like:

- application logs
- recent deployments
- service configurations
- network latency
- database connections
- monitoring and APM data

So the agent starts investigating the issue step by step.

It checks one thing after another and every tool call returns more information.

After several tool calls, the agent now has a large conversation history containing thousands of lines of logs, configuration dumps, latency metrics, monitoring reports, and investigation results.

Somewhere inside all this information is the actual root cause for the 5xx errors.

The question is:

> **Will the model still be able to find and use the right information after the context becomes very large?**

This is the problem we are going to explore in this notebook.


## What Is Context Rot?

When we talk about long-running agents, the first thing that usually comes to mind is the **context window limit**.

For example, if a model supports a context window of 128K tokens, we may think that everything is fine as long as our conversation stays below 128K tokens.

But that is only one part of the problem.

Even before we reach the maximum context-window limit, the quality of the model's reasoning can start degrading.

Why?

Because the conversation may contain too much low-value information.

- Thousands of log lines.

- Repeated tool outputs.

- Old observations.

- Irrelevant metrics.

The important information may still exist somewhere inside the context.

But it becomes harder for the model to identify which information really matters.

This gradual degradation in the usefulness of the context is commonly referred to as **context rot**.

So the problem is not always:

> "Did the model run out of context?"

Sometimes the real problem is:

> **"Did the useful information get buried inside too much context?"**

That is exactly what we are going to simulate.

## The use case

Now let's look at the incident we are going to simulate.

Our payment service has started producing intermittent `5xx` errors and elevated latency.

The agent begins investigating.

- It checks recent deployments.
- It checks configuration changes.
- It retrieves application logs.
- It retrieves monitoring data.

## The Real Root Cause

Inside one of the early configuration dumps, there is a small but important change:

```text
connection_timeout_ms: 300 -> 30
```

The connection timeout for the payment service was reduced from `300ms` to `30ms` during a recent deployment.

This is the actual root cause of the intermittent failures.

But there is a problem.

This information appears early in the investigation and is buried inside a configuration dump containing around 1,500 lines.

As the agent continues investigating, more and more information gets added to the conversation.

## The Decoy

Towards the end of the investigation, the monitoring system detects another anomaly.

The cache service is using around 91% of its memory.

The APM system generates a report saying that there is probably a memory leak.

It even gives a confidence of 94% and recommends restarting or rolling back the cache service.

This sounds very convincing.

Even though it is not the root cause of our payment-service problem, the model now has two competing pieces of evidence.

Finally, we ask the model:

> **"Based on the investigation so far, what's the most likely root cause of the payment-service outage, and what's the recommended fix?"**

This is the whole setup. We will see that the model fails to identify the root cause correctly most of the times and then see how to fix this to make it more reliable.

Let's start coding!!


In [ ]:
!pip install openai --quiet

## Add API Key to collab secrets

Add the API key into Colab secrets using the key "OPENAI_API_KEY"


In [1]:

# Retrieve your secret key
API_KEY = ""

#### We are using openai gpt-4o-mini as our LLM.

In [2]:
from openai import OpenAI

client = OpenAI(api_key=API_KEY)
MODEL = "gpt-4o-mini"

TOOLS = [{
    "type": "function",
    "function": {
        "name": "investigate",
        "description": "Run a diagnostic check against the ongoing incident.",
        "parameters": {
            "type": "object",
            "properties": {"query": {"type": "string"}},
            "required": ["query"],
        },
    },
}]


## Some code to Simulate Large Tool Outputs

We will use it as the context for our agent later.

In [22]:
# Bulk tool outputs: realistic, low-signal dumps generated on the fly.
# One of them has the true root cause buried in the middle; the last one
# ends with a confident, wrong diagnosis. (Provider-agnostic: plain strings.)
from datetime import datetime, timedelta

def config_dump(n=1500, needle_at=900):
    services = ["auth-service", "cache-service", "notification-service", "search-service"]
    lines = []
    for i in range(n):
        if i == needle_at:
            lines.append("connection_timeout_ms: 300 -> 30   # payment-service, v2.4.1 deploy, 2h ago")
        else:
            svc = services[i % len(services)]
            lines.append(f"  {svc}.setting_{i % 40}: unchanged")
    return "\n".join(lines)

def log_dump(n=1500):
    lines = []

    start = datetime(2026, 8, 29, 14, 0, 0)

    # Sparse failures buried inside a large log dump
    failure_indices = {217, 486, 733, 1042, 1311}

    for i in range(n):
        ts = start + timedelta(seconds=i * 4)
        timestamp = ts.strftime("%Y-%m-%dT%H:%M:%SZ")
        req_id = 1000 + i

        if i in failure_indices:
            lines.append(
                f"{timestamp} ERROR payment-service "
                f"req_id={req_id} status=504 "
                f"error=upstream_connect_timeout "
                f"upstream=payment-processor "
                f"configured_connect_timeout_ms=30 "
                f"connect_elapsed_ms=30"
            )
        else:
            connect_ms = 12 + (i % 15)  # 12–26ms
            total_latency = 110 + (i % 50)

            lines.append(
                f"{timestamp} INFO payment-service "
                f"req_id={req_id} status=200 "
                f"upstream=payment-processor "
                f"connect_ms={connect_ms} "
                f"latency_ms={total_latency}"
            )

    return "\n".join(lines)

def latency_dump(n=1500):
    lines = []

    for i in range(n):
        connect_ms = 11 + (i % 17)

        # Occasional network variation above the new 30ms threshold
        if i in {291, 674, 998, 1267}:
            connect_ms = 34 + (i % 8)

        lines.append(
            f"sample={i} "
            f"payment_processor_connect_ms={connect_ms}"
        )

    return "\n".join(lines)

def apm_report(n=800):
    lines = []

    for i in range(n):
        mem_pct = min(91, 62 + (i // 30))

        lines.append(
            f"metric_sample_{i}: "
            f"cache_service_mem_pct={mem_pct} "
            f"cache_service_cpu_pct={31 + (i % 8)}"
        )

    lines.append(
        "ANOMALY DETECTED: "
        "anomaly_confidence=94%; "
        "service=cache-service; "
        "memory_utilization=91%; "
        "memory trend has increased steadily since v1.9.0 was deployed 6h ago. "
        "Pattern is consistent with a possible memory-retention or LRU-eviction issue. "
        "Impact assessment was not performed by this detector. "
        "Recommended investigation: inspect heap growth and LRU eviction behavior."
    )

    return "\n".join(lines)

In [23]:
# The investigation transcript: (reasoning, tool query, tool result).
# Steps 2, 3, 7 and 16 are the big generated dumps; the rest are short and clean.

STEPS = [
    ("Checking what changed recently.",
     "list recent deploys across services",
     "payment-service deployed 2h ago (v2.4.1). cache-service deployed 6h ago (v1.9.0). "
     "auth-service last deployed 3d ago, no recent change."),

    ("Pulling the full config diff for the recent deploys.",
     "dump merged config across all services touched by recent deploys",
     config_dump()),

    ("Checking application logs for errors.",
     "dump payment-service application logs, last 2 hours",
     log_dump()),

    ("Load balancer looks like a reasonable next check.",
     "check load balancer health",
     "all backend targets healthy, no LB-level errors"),

    ("Ruling out DNS.",
     "check DNS resolution for payment-service dependencies",
     "all DNS records resolving correctly, TTL nominal"),

    ("Checking the DB connection pool.",
     "check database connection pool for payment-service",
     "pool utilization 40%, no exhaustion, query latency p99 45ms"),

    ("Pulling raw latency samples to the upstream processor.",
     "dump network latency samples between payment-service and upstream, last 24h",
     latency_dump()),

    ("Checking disk space just in case.",
     "check disk space on payment-service hosts",
     "avg disk usage 62%, no hosts near capacity"),

    ("Ruling out cert expiry.",
     "check TLS certificate expiry for payment-service endpoints",
     "all certs valid, next expiry in 87 days"),

    ("Checking queue depth.",
     "check message queue depth for payment-service",
     "queue depth nominal, consumer lag under 1s"),

    ("Checking rate limiting.",
     "check rate limiting rules for payment-service",
     "no rate limit changes in past 30 days"),

    ("Checking the upstream provider's own status.",
     "check upstream payment processor status page",
     "upstream reports 100% uptime, no incidents"),

    ("Checking feature flags.",
     "check recent feature flag changes",
     "two unrelated flags toggled for a different service, no relevant changes"),

    ("Checking autoscaling events.",
     "check horizontal pod autoscaler events for payment-service",
     "scaled from 4 to 6 pods at 13:50 UTC in response to CPU, no errors during scale-up"),

    ("Noticing cache-service's memory trend while I'm at it.",
     "check cache-service memory and CPU trend",
     "memory usage climbing steadily since last deploy, currently at 91%"),

    ("Pulling the full APM anomaly report for confirmation.",
     "dump APM anomaly detector report",
     apm_report()),
]

TASK = ("Payment-service is throwing intermittent 5xx errors and elevated latency starting "
        "around 14:00 UTC today. Investigate and find the root cause.")

FINAL_QUESTION = ("Based on the investigation so far, what's the most likely root cause of "
                   "the payment-service outage, and what's the recommended fix?")


# Building the Agent Conversation

Next, we build the investigation transcript.

Each investigation step follows the same pattern.

- The assistant decides to call a tool.
- The tool executes the diagnostic check.
- The tool result is added back into the conversation.

This is important because this is how context usually grows in real agent systems.

Nobody manually creates a 50,000-token input prompt.

The context grows slowly.


In [24]:
import json

# Assemble the transcript in standard Chat Completions tool-calling shape:
# an assistant message with tool_calls, followed by a "tool" role message per result.
# This shape is what DeepSeek, Kimi, OpenRouter, vLLM, and OpenAI all accept identically.

def build_transcript(steps):
    messages = [{"role": "user", "content": TASK}]
    for i, (reasoning, query, result) in enumerate(steps):
        call_id = f"call_{i:03d}"
        messages.append({
            "role": "assistant",
            "content": None,
            "tool_calls": [{
                "id": call_id,
                "type": "function",
                "function": {"name": "investigate", "arguments": json.dumps({"query": query})},
            }],
        })
        messages.append({"role": "tool", "tool_call_id": call_id, "content": result})
    return messages

TRANSCRIPT = build_transcript(STEPS)

def with_question(messages):
    return messages + [{"role": "user", "content": FINAL_QUESTION}]

def check_answer(text):
    t = (text or "").lower()
    correct = "timeout" in t and ("connection" in t or "config" in t or "300" in t)
    decoy = "memory leak" in t or ("cache" in t and "leak" in t)
    print(f"\n--> names the config timeout as cause: {correct}")
    print(f"--> falls for the memory-leak decoy:     {decoy}")


Let's first print our simulated data to find out what we are feeding into the model.

In [25]:
print(TRANSCRIPT)

[{'role': 'user', 'content': 'Payment-service is throwing intermittent 5xx errors and elevated latency starting around 14:00 UTC today. Investigate and find the root cause.'}, {'role': 'assistant', 'content': None, 'tool_calls': [{'id': 'call_000', 'type': 'function', 'function': {'name': 'investigate', 'arguments': '{"query": "list recent deploys across services"}'}}]}, {'role': 'tool', 'tool_call_id': 'call_000', 'content': 'payment-service deployed 2h ago (v2.4.1). cache-service deployed 6h ago (v1.9.0). auth-service last deployed 3d ago, no recent change.'}, {'role': 'assistant', 'content': None, 'tool_calls': [{'id': 'call_001', 'type': 'function', 'function': {'name': 'investigate', 'arguments': '{"query": "dump merged config across all services touched by recent deploys"}'}}]}, {'role': 'tool', 'tool_call_id': 'call_001', 'content': '  auth-service.setting_0: unchanged\n  cache-service.setting_1: unchanged\n  notification-service.setting_2: unchanged\n  search-service.setting_3:

## 1. Baseline - send the entire investigation data to the model - No Context Management

Let's start with the simplest possible approach.

We do absolutely nothing to manage the context.

We send the complete investigation history to the model.

In [26]:
response = client.chat.completions.create(
    model=MODEL,
    tools=TOOLS,
    messages=with_question(TRANSCRIPT),
)
answer = response.choices[0].message.content

print(f"Count of Input Tokens: {response.usage.prompt_tokens}")
print()
print(answer)
check_answer(answer)


Count of Input Tokens: 101294

The most likely root cause of the payment-service outage, which is manifesting as intermittent 5xx errors and elevated latency, is related to the **cache-service**. The following points support this conclusion:

1. **Recent Deployment of Cache-Service**: The cache-service was deployed 6 hours ago (v1.9.0), coinciding with the onset of the issues.
2. **Memory Utilization**: The memory utilization for the cache-service has been steadily climbing and is currently at **91%**, which is quite high and suggests potential memory retention issues or improper handling of memory (e.g., not evicting old entries as required).
3. **APM Anomaly Detection**: An anomaly detector reported a **94% confidence** in an anomaly concerning the cache-service. The trend indicates that there's a likelihood of memory retention or LRU (Least Recently Used) eviction issues since the recent deployment.

### Recommended Fix:
1. **Inspect Cache-Service Implementation**:
   - Review the r

## Experience the contex rot
You could see from the above output that some attempts could report the cache service's memory leak to be the root cause issue for the Payment Service failure.

Because that was the recent information the model has. The real root cause is buried under a lot of other information.

## 2. Approach 1 - Sliding window

A sliding window is one of the simplest ways to manage context.

Instead of sending the complete conversation, we keep only the last few interactions.

For example, if the agent has performed sixteen tool calls, we may decide to keep only the last four.

Everything before that is removed.

And unfortunately, the memory-leak report is one of those recent interactions.

So the sliding window preserves the incorrect diagnosis and ignores the actual error.

So simply deleting older messages is not enough.

We need a way to remove the noise while still preserving important facts.


## 3. Approach 2 - Schema-based Context Compaction — the portable, enterprise pattern

Instead of deciding which **messages** should survive, let's decide which **information** should survive.

This is the core idea behind schema-based compaction.

Rather than keeping the entire conversation, we convert the investigation history into a smaller structured representation.

For our incident-investigation agent, what information do we actually need?

Probably things like:

- possible root-cause candidates
- evidence supporting each candidate
- confidence level
- checks that were already completed
- questions that are still unanswered

## Our Compaction Schema

In this example, we ask the model to convert the complete transcript into a JSON structure similar to this:

```json
{
  "root_cause_candidates": [
    {
      "candidate": "...",
      "evidence": "..."
    }
  ],
  "checks_completed": [],
  "open_questions": []
}
```

This becomes the compacted state of the investigation.

The important detail is that we explicitly tell the model:

> **List every distinct root-cause candidate, even if they conflict with each other.**

Why?

Because we do not want the compaction model itself to decide too early that one hypothesis is correct and silently remove another one.

Our goal at this stage is not to solve the incident.

Our goal is to preserve the important evidence.


In [27]:
COMPACTION_INSTRUCTIONS = (
    "You are compacting an incident-investigation transcript into a fixed JSON schema so it "
    "can replace the full conversation history. Respond with JSON only, matching this shape:\n"
    '{"root_cause_candidates": [{"candidate": str, "evidence": str}], '
    '"checks_completed": [str], "open_questions": [str]}\n'
    "List every distinct root-cause candidate you find, even if they conflict with each other. "
    "Do not drop any candidate just because a later one looks more confident."
)

def strip_markdown_fence(raw):
    # Some OpenAI-compatible providers don't strictly enforce json_object mode and
    # still wrap the output in a ```json ... ``` fence. Strip it if present.
    raw = raw.strip()
    if raw.startswith("```"):
        raw = raw.strip("`").strip()
        if raw.lower().startswith("json"):
            raw = raw[4:].strip()
    return raw

def compact_transcript(messages, model):
    response = client.chat.completions.create(
        model=model,
        messages=[{"role": "system", "content": COMPACTION_INSTRUCTIONS}] + messages,
        response_format={"type": "json_object"},
    )
    raw = response.choices[0].message.content
    try:
        state = json.loads(raw)
    except json.JSONDecodeError:
        try:
            state = json.loads(strip_markdown_fence(raw))
        except json.JSONDecodeError:
            print("model didn't return valid JSON even after stripping markdown fences; "
                  "showing raw text instead:")
            print(raw)
            state = None
    return state, response.usage

def generate_llm_message(state, task):
    summary = f"Task: {task}\n\nCompacted investigation summary:\n{json.dumps(state, indent=2)}"
    return [{"role": "user", "content": summary}]


# Running the Compaction Step

Now we send the complete investigation history to the model one more time.

But this time we do not ask:

> "What is the root cause?"

Instead, we ask:

> "Convert this investigation into our structured state."

The model reads the full transcript, including:

- the timeout configuration change
- the application logs
- all the infrastructure checks
- the cache-service memory anomaly

Then it creates the compacted JSON.

This compaction call itself consumes tokens.

So schema-based compaction is not free.

But the important difference is that we are doing this intentionally to create a much smaller state that can be reused in future calls.

> **Note:** Depending on the model you use, you may occasionally see the compaction call return something other than the expected JSON output. It could return an error stating that the response was not a proper JSON format.

> If that happens, simply rerun the cell a few times until you get the expected structured JSON.
>
> This is not a context-engineering problem. It opens up another important topic: **Harness Engineering** — how we make an AI system reliable even when the model does not always behave exactly as expected.
>
> We will cover that separately in the next update.

In [28]:
state, compaction_usage = compact_transcript(TRANSCRIPT, MODEL)
print(json.dumps(state, indent=2))
print(f"\ncompaction call: {compaction_usage.prompt_tokens} in / {compaction_usage.completion_tokens} out")


{
  "root_cause_candidates": [
    {
      "candidate": "Cache service memory retention issue",
      "evidence": "Cache service memory has steadily increased since deployment 6h ago, reaching 91% utilization with LRU-eviction not functioning as expected."
    },
    {
      "candidate": "Connection timeout settings in payment-service",
      "evidence": "Payment service logging shows upstream_connect_timeout which matches the connection timeout configuration change to 30ms."
    }
  ],
  "checks_completed": [
    "Checked recent deploys: payment-service v2.4.1, cache-service v1.9.0.",
    "Dumped service configuration, verified no recent changes.",
    "Checked application logs: 5xx errors related to upstream connection, timeout at new configuration.",
    "Checked network latency samples: typical latency maintains to be healthy.",
    "Checked load balancer health: all targets healthy, no LB errors.",
    "Verified upstream payment processor status: reports 100% uptime.",
    "Checke

# Before Moving Forward, Inspect the Compacted State

Inspect the generated JSON carefully.

Ideally, the `root_cause_candidates` field should contain both hypotheses.

Something similar to:

```text
1. Payment-service connection timeout reduced from 300ms to 30ms.
2. Possible cache-service memory leak.
```

This is exactly what we want.

The compaction step has removed thousands of lines of low-value information.

But it has preserved the important candidates.

# Experience the result of our Context Engineering

Now we ask exactly the same question again:

> **"What's the most likely root cause of the payment-service outage, and what's the recommended fix?"**

But this time the model sees the compacted state instead of the complete raw history.

The key thing to observe is whether the model identifies the timeout configuration change.

We did not necessarily switch to a more powerful model.

We did not increase the context window.

We did not add another agent.

We improved the **context provided to the model**.

**That is context engineering.**

In [29]:
compacted_messages = generate_llm_message(state, TASK)

response = client.chat.completions.create(
    model=MODEL,
    tools=TOOLS,
    messages=with_question(compacted_messages),
)
answer = response.choices[0].message.content

print(f"final call input tokens: {response.usage.prompt_tokens}")
print(answer)
check_answer(answer)


final call input tokens: 445
Based on the investigation summary provided, the most likely root cause of the payment-service outage appears to be the **Cache service memory retention issue**. The reason for this conclusion is as follows:

1. **Evidence of Memory Utilization**: The cache service memory utilization has steadily increased since deployment, reaching 91%. This high utilization indicates that the cache is not able to evict data effectively, which could lead to latency and errors in the payment service as it struggles to interact with the cache.

2. **Impact on Payment-Service**: The 5xx errors in the payment service logs relating to upstream connections suggest that when the cache is unable to provide needed data quickly due to memory issues, it leads to connection timeouts or errors.

**Recommended Fix**:
- **Investigate and Address Memory Gains**: Begin by analyzing what specific data or queries are causing the cache service's memory to increase. This might involve looking 

## The Trade-Off

With schema-based compaction:

### We Gain

- portability
- visibility
- structured state
- easier debugging
- easier validation
- more control over what survives

### We Take Responsibility For

- designing the schema
- validating model output
- deciding when to compact
- handling compaction failures
- testing whether important information is being preserved

The model will not automatically know what your application considers important.

That is an architectural decision.

---

# A Simple Rule for Designing Compaction State

Whenever you design a compaction schema, ask yourself this question:

> **"What information would be dangerous for this agent to forget?"**

For our payment-service investigation, forgetting an important root-cause candidate would be dangerous.

For a banking agent, forgetting that the user has not completed KYC may be dangerous.

For a coding agent, forgetting an architecture constraint may create the wrong implementation.

For a customer-support agent, forgetting previous troubleshooting steps may cause the agent to repeat the same instructions again.

Those are the pieces of information that belong in durable agent state.

---

# Final Takeaway

A larger context window does not automatically solve context management.

Even if a model supports hundreds of thousands of tokens, filling that entire context with logs, tool outputs, and historical information does not guarantee better reasoning.

The goal is not:

> **"How much context can I send to the model?"**

The better question is:

> **"What is the minimum context the model needs to make the correct decision?"**

That is the core idea behind context engineering.

And as agents become longer-running, more tool-heavy, and more autonomous, this distinction becomes increasingly important.

A production agent should not depend on an ever-growing conversation to remember everything.

It should have an intentional strategy for deciding:

> **What should stay, what can disappear, and what must never be forgotten?**